# MedSecure-XR — Kriptografi + Watermarking pada Citra Rontgen Dada
**Tugas Akhir Keamanan Data · Capstone 2026 cabang Kriptografi dan Keamanan Data**

Dua metode yang digabung:
1. **Watermarking**: watermark *fragile* LSB yang disisipkan **hanya di RONI** (latar gelap), sehingga area diagnosis (ROI) tidak berubah sama sekali.
2. **Kriptografi**: AES-256-GCM (kerahasiaan EPR), HMAC-SHA256 per blok (lokalisasi manipulasi), Ed25519 (tanda tangan dokter), AES-256-SIV (enkripsi seluruh citra saat dikirim).

Alur pengirim: cover → deteksi RONI → [EPR terenkripsi | tag blok | tanda tangan] → sisip LSB di RONI → enkripsi AES-SIV.
Alur penerima: dekripsi AES-SIV → ekstraksi watermark → verifikasi tanda tangan → peta manipulasi → dekripsi EPR.

Jalankan dari atas ke bawah: **Runtime → Run all**.

## 0. Install & import

In [ ]:
!pip install -q cryptography scikit-image openpyxl

import base64
import hashlib
import hmac
import io
import json
import os
import struct
import time
from dataclasses import dataclass
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from scipy import ndimage
from cryptography.exceptions import InvalidSignature, InvalidTag
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PrivateKey,
    Ed25519PublicKey,
)
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, AESSIV
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import shutil, tracemalloc
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
print("Import OK")

## 1. Konfigurasi
Ukuran citra: tinggi 512 piksel, lebar mengikuti rasio asli (dibulatkan ke kelipatan 32), **tanpa center-crop**.
Center-crop sempat membuat paru menempel di tepi gambar sehingga terbaca sebagai latar (RONI) — kesalahan itu dihindari di sini.

In [ ]:
TARGET_H = 512          # output height; width follows the original aspect ratio
RONI_BLOCK = 16         # block size used to detect background (RONI)
TAMPER_BLOCK = 32       # block size of the tamper-localization grid
BG_THRESHOLD = 40       # a RONI block has all pixels (LSB ignored) below this value
RONI_MARGIN = 1         # safety margin (in RONI blocks) between ROI and RONI
TAG_BYTES = 2           # truncated HMAC tag per tamper block (16 bit)
MAGIC = b"MSX1"
PKG_MAGIC = b"MSXP"
SIG_LEN = 64
SIV_TAG_LEN = 16
GCM_NONCE_LEN = 12
LEN_BITS = 32


class WatermarkError(Exception):
    """Watermark missing, damaged, or does not fit."""

MODE = "siv"     # "siv" (dipakai) | "ctr" | "cbc" (pembanding)
REPEAT = 5       # ulangan untuk rata-rata waktu proses
OUT_DIR = Path("output"); OUT_DIR.mkdir(exist_ok=True)

# Data EPR (data fiktif untuk pengujian). Urutan mengikuti tabel tugas.
FIELDS = [
    ("Nama RS", "RS UM Sejahtera"),
    ("Tanggal", "4 November 2026"),
    ("No. Rekam Medis", "RM-2026-001234"),
    ("Nama Pasien", "Nabihah Alya Az Zahidah"),
    ("TTL", "24 Januari 2026"),
    ("Jam", "21:57"),
    ("Jenis Kelamin", "Female"),
    ("Golongan Darah", "B"),
    ("Alamat Pasien", "Jalan Semarang"),
    ("Nama Dokter", "Taharo Hafwenia"),
    ("Jenis Pemeriksaan", "Rontgen Thorax PA"),
]
# jumlah field untuk payload 1..10 (payload 4 menambah Nama Pasien + TTL)
FIELD_COUNTS = [1, 2, 3, 5, 6, 7, 8, 9, 10, 11]

## 2. Upload citra rontgen (cover)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()          # pilih file X-ray
    COVER_PATH = next(iter(uploaded))
except ImportError:
    COVER_PATH = "sample_xray.jpg"     # di luar Colab: isi path manual
print("Cover:", COVER_PATH)

## 3. Library inti
### 3.1 Kunci (AES-SIV, AES-GCM, HMAC, Ed25519)

In [ ]:
@dataclass
class KeyBundle:
    k_img: bytes                 # 64 B, AES-256-SIV (image transport)
    k_epr: bytes                 # 32 B, AES-256-GCM (patient record)
    k_wm: bytes                  # 32 B, HMAC key (block tags + embedding order)
    signing_key: Ed25519PrivateKey  # doctor's private key

    @classmethod
    def generate(cls) -> "KeyBundle":
        return cls(AESSIV.generate_key(512), AESGCM.generate_key(256), os.urandom(32),
                   Ed25519PrivateKey.generate())

    @property
    def public_key(self) -> Ed25519PublicKey:
        return self.signing_key.public_key()

    def public_bytes(self) -> bytes:
        return self.public_key.public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)

    def fingerprint(self) -> str:
        return hashlib.sha256(self.public_bytes()).hexdigest()[:16]

    def to_json(self) -> str:
        sk = self.signing_key.private_bytes(serialization.Encoding.Raw, serialization.PrivateFormat.Raw,
                                            serialization.NoEncryption())
        d = {"k_img": self.k_img, "k_epr": self.k_epr, "k_wm": self.k_wm, "ed25519_sk": sk,
             "ed25519_pk": self.public_bytes()}
        return json.dumps({k: base64.b64encode(v).decode() for k, v in d.items()}, indent=2)

    @classmethod
    def from_json(cls, text: str) -> "KeyBundle":
        d = {k: base64.b64decode(v) for k, v in json.loads(text).items()}
        return cls(d["k_img"], d["k_epr"], d["k_wm"], Ed25519PrivateKey.from_private_bytes(d["ed25519_sk"]))


def load_public_key(raw: bytes) -> Ed25519PublicKey:
    return Ed25519PublicKey.from_public_bytes(raw)

### 3.2 Prapemrosesan & deteksi RONI otomatis

In [ ]:
def load_cover(src, target_h: int = TARGET_H, grid: int = TAMPER_BLOCK):
    """Grayscale, resize keeping the aspect ratio (width rounded to a multiple of `grid`)."""
    img = Image.open(src).convert("L")
    w, h = img.size
    new_w = max(grid, int(round(w * target_h / h / grid)) * grid)
    arr = np.array(img.resize((new_w, target_h), Image.LANCZOS), dtype=np.uint8)
    return arr, (w, h)


def roni_mask(img: np.ndarray, block: int = RONI_BLOCK, thr: int = BG_THRESHOLD,
              margin: int = RONI_MARGIN) -> np.ndarray:
    """Boolean mask of RONI pixels.

    A block is background when every pixel (LSB ignored) is below `thr` and the
    dark region is connected to the image border (true air/collimation, not the
    lungs). Ignoring the LSB makes the mask identical before and after embedding.
    """
    h, w = img.shape
    base = img & 0xFE
    bmax = base.reshape(h // block, block, w // block, block).max(axis=(1, 3))
    lab, _ = ndimage.label(bmax < thr)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    roni_b = np.isin(lab, border[border > 0])
    if margin:
        roni_b = ~ndimage.binary_dilation(~roni_b, iterations=margin)
    return np.kron(roni_b, np.ones((block, block), dtype=bool)).astype(bool)


def canonical(img: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Image with RONI LSBs cleared: identical before and after embedding."""
    c = img.copy()
    c[mask] &= 0xFE
    return c


def embedding_order(mask: np.ndarray, k_wm: bytes) -> np.ndarray:
    """Key-dependent pseudo-random order of RONI pixel indices."""
    idx = np.flatnonzero(mask.ravel())
    seed = int.from_bytes(hmac.new(k_wm, b"embed-order", hashlib.sha256).digest()[:8], "big")
    return idx[np.random.default_rng(seed).permutation(idx.size)]

### 3.3 Blok kriptografi: tag HMAC per blok, pesan tanda tangan, enkripsi citra

In [ ]:
def block_tags(canon: np.ndarray, k_wm: bytes, tb: int = TAMPER_BLOCK) -> bytes:
    h, w = canon.shape
    out = bytearray()
    for by in range(h // tb):
        for bx in range(w // tb):
            blk = canon[by * tb:(by + 1) * tb, bx * tb:(bx + 1) * tb]
            msg = struct.pack(">HHHH", by, bx, h, w) + blk.tobytes()
            out += hmac.new(k_wm, msg, hashlib.sha256).digest()[:TAG_BYTES]
    return bytes(out)


def signature_message(canon: np.ndarray, body_wo_sig: bytes) -> bytes:
    return b"MedSecure-XR/v1" + hashlib.sha256(canon.tobytes()).digest() + hashlib.sha256(body_wo_sig).digest()


def aes_encrypt(img: np.ndarray, key: bytes, mode: str = "siv", iv: bytes | None = None):
    pt = img.tobytes()
    if mode == "siv":
        out = AESSIV(key).encrypt(pt, None)
        meta, body = out[:SIV_TAG_LEN], out[SIV_TAG_LEN:]
    else:
        k = key[:32]
        iv = iv or os.urandom(16)
        m = modes.CTR(iv) if mode == "ctr" else modes.CBC(iv)
        enc = Cipher(algorithms.AES(k), m).encryptor()
        body = enc.update(pt) + enc.finalize()
        meta = iv
    return np.frombuffer(body, dtype=np.uint8).reshape(img.shape).copy(), meta


def aes_decrypt(cipher_img: np.ndarray, meta: bytes, key: bytes, mode: str = "siv") -> np.ndarray:
    body = cipher_img.tobytes()
    if mode == "siv":
        pt = AESSIV(key).decrypt(meta + body, None)  # raises InvalidTag
    else:
        m = modes.CTR(meta) if mode == "ctr" else modes.CBC(meta)
        dec = Cipher(algorithms.AES(key[:32]), m).decryptor()
        pt = dec.update(body) + dec.finalize()
    return np.frombuffer(pt, dtype=np.uint8).reshape(cipher_img.shape).copy()

### 3.4 Struktur payload watermark

In [ ]:
def _header(h: int, w: int) -> bytes:
    return MAGIC + struct.pack(">BBBHH", 1, TAMPER_BLOCK, TAG_BYTES, h, w)


HEADER_LEN = len(MAGIC) + 7


def build_body(canon: np.ndarray, epr_text: str, keys: KeyBundle, times: dict | None = None) -> bytes:
    times = times if times is not None else {}
    h, w = canon.shape
    hdr = _header(h, w)

    s = time.perf_counter()
    nonce = os.urandom(GCM_NONCE_LEN)
    ct = AESGCM(keys.k_epr).encrypt(nonce, epr_text.encode("utf-8"), hdr)
    times["epr_encrypt"] = time.perf_counter() - s

    s = time.perf_counter()
    tags = block_tags(canon, keys.k_wm)
    times["block_tags"] = time.perf_counter() - s

    body = hdr + nonce + struct.pack(">H", len(ct)) + ct + tags
    s = time.perf_counter()
    sig = keys.signing_key.sign(signature_message(canon, body))
    times["sign"] = time.perf_counter() - s
    return body + sig


def body_to_bits(body: bytes) -> np.ndarray:
    return np.unpackbits(np.frombuffer(len(body).to_bytes(LEN_BITS // 8, "big") + body, dtype=np.uint8))


def parse_body(body: bytes) -> dict:
    if len(body) < HEADER_LEN + GCM_NONCE_LEN + 2 + SIG_LEN or body[:4] != MAGIC:
        raise WatermarkError("Watermark tidak ditemukan / rusak (magic tidak cocok)")
    ver, tb, tag_bytes, h, w = struct.unpack(">BBBHH", body[4:HEADER_LEN])
    p = HEADER_LEN
    nonce = body[p:p + GCM_NONCE_LEN]; p += GCM_NONCE_LEN
    (ct_len,) = struct.unpack(">H", body[p:p + 2]); p += 2
    ct = body[p:p + ct_len]; p += ct_len
    n_tags = (h // tb) * (w // tb)
    tags = body[p:p + n_tags * tag_bytes]; p += n_tags * tag_bytes
    sig = body[p:p + SIG_LEN]
    if len(sig) != SIG_LEN or len(tags) != n_tags * tag_bytes:
        raise WatermarkError("Struktur watermark tidak lengkap")
    return {"version": ver, "tb": tb, "tag_bytes": tag_bytes, "h": h, "w": w, "header": body[:HEADER_LEN],
            "nonce": nonce, "ct": ct, "tags": tags, "sig": sig, "body_wo_sig": body[:-SIG_LEN]}


def capacity_bits(mask: np.ndarray) -> int:
    return int(mask.sum())


def overhead_bytes(shape) -> int:
    h, w = shape
    n_tags = (h // TAMPER_BLOCK) * (w // TAMPER_BLOCK)
    return LEN_BITS // 8 + HEADER_LEN + GCM_NONCE_LEN + 2 + 16 + n_tags * TAG_BYTES + SIG_LEN


def max_epr_bytes(cover: np.ndarray) -> int:
    return capacity_bits(roni_mask(cover)) // 8 - overhead_bytes(cover.shape)

### 3.5 Pengirim: `protect()`

In [ ]:
def protect(cover: np.ndarray, epr_text: str, keys: KeyBundle, mode: str = "siv") -> dict:
    """Embed signed+encrypted EPR in RONI, then encrypt the whole image."""
    t = {}
    s = time.perf_counter()
    mask = roni_mask(cover)
    canon = canonical(cover, mask)
    order = embedding_order(mask, keys.k_wm)
    t["roni_mask"] = time.perf_counter() - s

    body = build_body(canon, epr_text, keys, t)
    bits = body_to_bits(body)
    if bits.size > order.size:
        raise WatermarkError(f"Kapasitas RONI tidak cukup: butuh {bits.size} bit, tersedia {order.size} bit")

    s = time.perf_counter()
    flat = cover.ravel().copy()
    pos = order[:bits.size]
    flat[pos] = (flat[pos] & 0xFE) | bits
    wm = flat.reshape(cover.shape)
    t["embed"] = time.perf_counter() - s

    s = time.perf_counter()
    enc, meta = aes_encrypt(wm, keys.k_img, mode)
    t["img_encrypt"] = time.perf_counter() - s
    return {"watermarked": wm, "encrypted": enc, "meta": meta, "mode": mode, "mask": mask,
            "bits": bits, "body_len": len(body), "capacity_bits": int(order.size), "times": t}

### 3.6 Penerima: ekstraksi, verifikasi, peta manipulasi

In [ ]:
def extract_body(img: np.ndarray, k_wm: bytes):
    mask = roni_mask(img)
    order = embedding_order(mask, k_wm)
    if order.size < LEN_BITS:
        raise WatermarkError("RONI was not found in the image")
    lsb = (img.ravel()[order] & 1).astype(np.uint8)
    n = int.from_bytes(np.packbits(lsb[:LEN_BITS]).tobytes(), "big")
    if n <= 0 or LEN_BITS + 8 * n > order.size:
        raise WatermarkError("Panjang watermark tidak valid (watermark rusak)")
    bits = lsb[:LEN_BITS + 8 * n]
    return np.packbits(bits[LEN_BITS:]).tobytes(), bits, mask


def verify_image(img: np.ndarray, keys: KeyBundle, public_key: Ed25519PublicKey) -> dict:
    """Verify a plaintext (decrypted) image: watermark, signature, tamper map, EPR."""
    t = {}
    r = {"watermark_found": False, "signature_ok": False, "epr_ok": False, "epr": None,
         "tamper_map": None, "n_tampered": None, "bits": None, "times": t}
    s = time.perf_counter()
    try:
        body, bits, mask = extract_body(img, keys.k_wm)
        p = parse_body(body)
    except WatermarkError as e:
        t["extract"] = time.perf_counter() - s
        r["message"] = str(e)
        r["status"] = "TAMPERED"
        return r
    t["extract"] = time.perf_counter() - s
    r["watermark_found"] = True
    r["bits"] = bits

    canon = canonical(img, mask)
    s = time.perf_counter()
    try:
        public_key.verify(p["sig"], signature_message(canon, p["body_wo_sig"]))
        r["signature_ok"] = True
    except InvalidSignature:
        pass
    t["verify_sig"] = time.perf_counter() - s

    s = time.perf_counter()
    if p["tb"] == TAMPER_BLOCK and (p["h"], p["w"]) == img.shape:
        fresh = block_tags(canon, keys.k_wm)
        gh, gw = p["h"] // p["tb"], p["w"] // p["tb"]
        a = np.frombuffer(fresh, dtype=">u2").reshape(gh, gw)
        b = np.frombuffer(p["tags"], dtype=">u2").reshape(gh, gw)
        r["tamper_map"] = a != b
        r["n_tampered"] = int(r["tamper_map"].sum())
    t["tamper_check"] = time.perf_counter() - s

    s = time.perf_counter()
    try:
        r["epr"] = AESGCM(keys.k_epr).decrypt(p["nonce"], p["ct"], p["header"]).decode("utf-8")
        r["epr_ok"] = True
    except InvalidTag:
        pass
    t["epr_decrypt"] = time.perf_counter() - s

    if r["signature_ok"] and r["epr_ok"] and r["n_tampered"] == 0:
        r["status"], r["message"] = "SAFE", "Authentic image, valid doctor signature, and intact EPR"
    elif r["n_tampered"]:
        r["status"] = "TAMPERED"
        r["message"] = f"{r['n_tampered']} changed {TAMPER_BLOCK}x{TAMPER_BLOCK}-pixel blocks detected"
    else:
        r["status"] = "TAMPERED"
        r["message"] = "Invalid signature or watermark data"
    return r


def receive(cipher_img: np.ndarray, meta: bytes, keys: KeyBundle, public_key: Ed25519PublicKey,
            mode: str = "siv") -> dict:
    s = time.perf_counter()
    try:
        dec = aes_decrypt(cipher_img, meta, keys.k_img, mode)
    except InvalidTag:
        return {"status": "REJECTED", "transit_ok": False, "image": None,
                "message": "AES-SIV decryption failed: wrong key or ciphertext altered in transit",
                "times": {"img_decrypt": time.perf_counter() - s}}
    t_dec = time.perf_counter() - s
    r = verify_image(dec, keys, public_key)
    r["times"] = {"img_decrypt": t_dec, **r["times"]}
    r["transit_ok"] = True
    r["image"] = dec
    return r

### 3.7 Paket transmisi (1 file)

In [ ]:
_MODES = {"siv": 0, "ctr": 1, "cbc": 2}


def pack_package(enc: np.ndarray, meta: bytes, mode: str = "siv") -> bytes:
    h, w = enc.shape
    return PKG_MAGIC + struct.pack(">BBHHB", 1, _MODES[mode], h, w, len(meta)) + meta + enc.tobytes()


def unpack_package(data: bytes):
    if data[:4] != PKG_MAGIC:
        raise ValueError("Bukan file paket MedSecure-XR")
    ver, m, h, w, ml = struct.unpack(">BBHHB", data[4:11])
    meta = data[11:11 + ml]
    body = data[11 + ml:]
    if len(body) != h * w:
        raise ValueError("Ukuran paket tidak sesuai")
    mode = {v: k for k, v in _MODES.items()}[m]
    return np.frombuffer(body, dtype=np.uint8).reshape(h, w).copy(), meta, mode

### 3.8 Metrik evaluasi

In [ ]:
def mse(a, b):
    return float(np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2))


def psnr(a, b):
    return float("inf") if mse(a, b) == 0 else float(peak_signal_noise_ratio(a, b, data_range=255))


def snr(a, b):
    """Signal-to-noise ratio (dB): power of the original image / power of the watermark change."""
    noise = mse(a, b)
    return float("inf") if noise == 0 else float(10 * np.log10(np.mean(a.astype(np.float64) ** 2) / noise))


def psnr_region(a, b, mask):
    d = (a.astype(np.float64) - b.astype(np.float64))[mask]
    m = float(np.mean(d ** 2)) if d.size else 0.0
    return float("inf") if m == 0 else float(10 * np.log10(255 ** 2 / m))


def ssim(a, b):
    return float(structural_similarity(a, b, data_range=255))


def entropy(img):
    h = np.bincount(img.ravel(), minlength=256).astype(np.float64)
    p = h[h > 0] / h.sum()
    return float(-np.sum(p * np.log2(p)))


def adjacent_correlation(img):
    x = img.astype(np.float64)
    pairs = {"H": (x[:, :-1], x[:, 1:]), "V": (x[:-1, :], x[1:, :]), "D": (x[:-1, :-1], x[1:, 1:])}
    return {k: float(np.corrcoef(a.ravel(), b.ravel())[0, 1]) for k, (a, b) in pairs.items()}


def npcr_uaci(c1, c2):
    a, b = c1.astype(np.int16), c2.astype(np.int16)
    return float(100 * np.mean(a != b)), float(100 * np.mean(np.abs(a - b) / 255))


def differential_test(img, key: bytes, mode: str = "siv"):
    """Change one pixel (+1 mod 256); encrypt both with the SAME key and SAME IV."""
    img2 = img.copy()
    r, c = img.shape[0] // 2, img.shape[1] // 2
    img2[r, c] = (int(img2[r, c]) + 1) % 256
    iv = os.urandom(16)
    c1, _ = aes_encrypt(img, key, mode, iv)
    c2, _ = aes_encrypt(img2, key, mode, iv)
    return npcr_uaci(c1, c2)


def ber(a, b):
    n, m = max(a.size, b.size), min(a.size, b.size)
    return (int(np.sum(a[:m] != b[:m])) + (n - m)) / n if n else 0.0


def nc(a, b):
    a, b = a.astype(np.float64), b.astype(np.float64)
    m = min(a.size, b.size)
    a, b = a[:m], b[:m]
    den = np.sqrt(np.sum(a * a) * np.sum(b * b))
    return float(np.sum(a * b) / den) if den else 0.0


def sha256_img(img) -> str:
    return hashlib.sha256(img.tobytes()).hexdigest()

### 3.9 Simulasi serangan & skor lokalisasi

In [ ]:
ATTACKS = {
    "hapus_lesi": "Remove lesion (fill area with surrounding average)",
    "copy_move": "Copy-move (paste another lung region)",
    "terangkan": "Brighten region (+40)",
    "blur": "Local blur",
    "teks": "Add text annotation",
    "noise": "Global Gaussian noise (sigma 2)",
    "jpeg": "Global JPEG compression (Q=90)",
}


def default_box(shape, frac=0.12):
    h, w = shape
    bh, bw = int(h * frac), int(w * frac)
    return (int(w * 0.30), int(h * 0.45), bw, bh)  # x, y, w, h (left lung area)


def attack(img: np.ndarray, kind: str, box=None, seed: int = 0) -> np.ndarray:
    a = img.copy()
    x, y, bw, bh = box or default_box(img.shape)
    sl = (slice(y, y + bh), slice(x, x + bw))
    if kind == "hapus_lesi":
        ring = img[max(0, y - 8):y + bh + 8, max(0, x - 8):x + bw + 8]
        a[sl] = np.uint8(round(float(ring.mean())))
    elif kind == "copy_move":
        sx = min(img.shape[1] - bw, x + int(img.shape[1] * 0.35))
        a[sl] = img[y:y + bh, sx:sx + bw]
    elif kind == "terangkan":
        a[sl] = np.clip(a[sl].astype(int) + 40, 0, 255).astype(np.uint8)
    elif kind == "blur":
        pil = Image.fromarray(a)
        region = pil.crop((x, y, x + bw, y + bh)).filter(ImageFilter.GaussianBlur(4))
        pil.paste(region, (x, y))
        a = np.array(pil)
    elif kind == "teks":
        pil = Image.fromarray(a)
        ImageDraw.Draw(pil).text((x, y), "NORMAL", fill=255)
        a = np.array(pil)
    elif kind == "noise":
        rng = np.random.default_rng(seed)
        a = np.clip(a.astype(float) + rng.normal(0, 2, a.shape), 0, 255).astype(np.uint8)
    elif kind == "jpeg":
        buf = io.BytesIO()
        Image.fromarray(a).save(buf, "JPEG", quality=90)
        a = np.array(Image.open(io.BytesIO(buf.getvalue())).convert("L"))
    else:
        raise ValueError(kind)
    return a


def block_ground_truth(orig: np.ndarray, attacked: np.ndarray, tb: int = TAMPER_BLOCK) -> np.ndarray:
    h, w = orig.shape
    d = orig != attacked
    return d.reshape(h // tb, tb, w // tb, tb).any(axis=(1, 3))


def localization_scores(pred: np.ndarray, gt: np.ndarray) -> dict:
    tp = int(np.sum(pred & gt)); fp = int(np.sum(pred & ~gt))
    fn = int(np.sum(~pred & gt)); tn = int(np.sum(~pred & ~gt))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "TPR": tp / (tp + fn) if tp + fn else float("nan"),
            "FPR": fp / (fp + tn) if fp + tn else 0.0}

### 3.10 Helper visual

In [ ]:
def overlay_mask(img: np.ndarray, mask: np.ndarray, color=(0, 170, 255), alpha=0.45) -> np.ndarray:
    rgb = np.stack([img] * 3, axis=-1).astype(np.float64)
    c = np.array(color, dtype=np.float64)
    rgb[mask] = (1 - alpha) * rgb[mask] + alpha * c
    return rgb.clip(0, 255).astype(np.uint8)


def tamper_overlay(img: np.ndarray, tamper_map: np.ndarray, tb: int = TAMPER_BLOCK) -> np.ndarray:
    full = np.kron(tamper_map, np.ones((tb, tb), dtype=bool)).astype(bool)
    return overlay_mask(img, full, color=(255, 40, 40), alpha=0.5)


def to_png_bytes(arr: np.ndarray) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, "PNG")
    return buf.getvalue()

## Tahap A — Persiapan
### A1. Load cover, deteksi RONI, kapasitas

In [ ]:
cover, orig_size = load_cover(COVER_PATH)
Image.fromarray(cover).save(OUT_DIR / "cover.png")      # lossless
MASK = roni_mask(cover)
ROI = ~MASK
N_PIXELS = cover.size
CAP_BITS = capacity_bits(MASK)
print(f"Ukuran asli {orig_size} -> cover {cover.shape[1]}x{cover.shape[0]} ({N_PIXELS} piksel)")
print(f"RONI: {CAP_BITS} piksel ({100*CAP_BITS/N_PIXELS:.1f}%) = kapasitas {CAP_BITS} bit")
print(f"Overhead tetap (header+nonce+tag GCM+tag blok+tanda tangan): {overhead_bytes(cover.shape)} byte")
print(f"Kapasitas maksimum EPR: {max_epr_bytes(cover)} byte")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(cover, cmap="gray", vmin=0, vmax=255); ax[0].set_title("Cover"); ax[0].axis("off")
ax[1].imshow(overlay_mask(cover, MASK)); ax[1].set_title("RONI terdeteksi (biru) = area penyisipan"); ax[1].axis("off")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_roni.png", dpi=150); plt.show()

### A2–A3. 10 payload EPR, ukuran & capacity (bpp)

In [ ]:
def build_payloads():
    return ["; ".join(f"{k}: {v}" for k, v in FIELDS[:n]) for n in FIELD_COUNTS]

payloads = build_payloads()
OVH = overhead_bytes(cover.shape)
rows_p = []
for i, p in enumerate(payloads, 1):
    epr_b = len(p.encode("utf-8"))
    total_bits = (OVH + epr_b) * 8
    rows_p.append({"payload": i, "fields": FIELD_COUNTS[i-1], "epr_bytes": epr_b, "epr_bits": epr_b * 8,
                   "total_embedded_bits": total_bits, "bpp_image": total_bits / N_PIXELS,
                   "roni_used_%": 100 * total_bits / CAP_BITS, "fits": total_bits <= CAP_BITS, "text": p})
df_payload = pd.DataFrame(rows_p)
assert df_payload["fits"].all(), "Ada payload yang tidak muat di RONI!"
df_payload.to_csv(OUT_DIR / "payloads.csv", index=False)
df_payload

### A4. Kunci (demo: dibangkitkan sekali, simpan aman — jangan dibagikan)

In [ ]:
KEYS = KeyBundle.generate()
(OUT_DIR / "keys_DEMO_JANGAN_DIBAGIKAN.json").write_text(KEYS.to_json())
print("Fingerprint kunci publik dokter:", KEYS.fingerprint())

## Tahap B — Loop 10 payload

In [ ]:
def run_once(payload):
    p = protect(cover, payload, KEYS, MODE)
    v = receive(p["encrypted"], p["meta"], KEYS, KEYS.public_key, MODE)
    return p, v

def peak_memory_kb(payload):
    tracemalloc.start(); run_once(payload)
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    return peak / 1024

run_once(payloads[0])   # warm-up
STAGES_P = ["roni_mask", "epr_encrypt", "block_tags", "sign", "embed", "img_encrypt"]
STAGES_V = ["img_decrypt", "extract", "verify_sig", "tamper_check", "epr_decrypt"]

rows, results = [], {}
for i, payload in enumerate(payloads, 1):
    acc = {k: [] for k in STAGES_P + STAGES_V}
    for _ in range(REPEAT):
        p, v = run_once(payload)
        for k in STAGES_P: acc[k].append(p["times"][k])
        for k in STAGES_V: acc[k].append(v["times"][k])
    wm, enc = p["watermarked"], p["encrypted"]
    cp, cc = adjacent_correlation(wm), adjacent_correlation(enc)
    npcr_pc, uaci_pc = npcr_uaci(wm, enc)
    npcr_d, uaci_d = differential_test(wm, KEYS.k_img, MODE)
    row = {
        "payload": i, "epr_bytes": len(payload.encode()), "embedded_bits": int(p["bits"].size),
        "bpp_image": p["bits"].size / N_PIXELS, "roni_used_%": 100 * p["bits"].size / CAP_BITS,
        "changed_px": int(np.sum(cover != wm)), "changed_px_ROI": int(np.sum((cover != wm) & ROI)),
        "MSE": mse(cover, wm), "PSNR_dB": psnr(cover, wm), "PSNR_ROI_dB": psnr_region(cover, wm, ROI),
        "SSIM": ssim(cover, wm), "SNR_dB": snr(cover, wm),
        "entropy_plain": entropy(wm), "entropy_cipher": entropy(enc),
        "corrH_plain": cp["H"], "corrV_plain": cp["V"], "corrD_plain": cp["D"],
        "corrH_cipher": cc["H"], "corrV_cipher": cc["V"], "corrD_cipher": cc["D"],
        "NPCR_plain_vs_cipher_%": npcr_pc, "UACI_plain_vs_cipher_%": uaci_pc,
        "NPCR_diff_1px_%": npcr_d, "UACI_diff_1px_%": uaci_d,
        "hash_match": sha256_img(v["image"]) == sha256_img(wm),
        "BER": ber(p["bits"], v["bits"]), "NC": nc(p["bits"], v["bits"]),
        "signature_ok": v["signature_ok"], "tampered_blocks": v["n_tampered"],
        "epr_match": v["epr"] == payload, "status": v["status"],
        **{f"t_{k}_ms": 1000 * np.mean(acc[k]) for k in STAGES_P + STAGES_V},
        "peak_mem_KB": peak_memory_kb(payload),
    }
    row["t_sender_ms"] = sum(row[f"t_{k}_ms"] for k in STAGES_P)
    row["t_receiver_ms"] = sum(row[f"t_{k}_ms"] for k in STAGES_V)
    rows.append(row)
    results[i] = {"p": p, "v": v}

    pdir = OUT_DIR / f"payload_{i:02d}"; pdir.mkdir(exist_ok=True)
    Image.fromarray(wm).save(pdir / "watermarked.png")
    Image.fromarray(enc).save(pdir / "encrypted_preview.png")
    (pdir / "package.msx").write_bytes(pack_package(enc, p["meta"], MODE))
    Image.fromarray(v["image"]).save(pdir / "decrypted.png")
    diff = (np.abs(cover.astype(np.int16) - wm.astype(np.int16)) * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(diff).save(pdir / "diff_x255.png")
    (pdir / "epr_extracted.txt").write_text(v["epr"], encoding="utf-8")
    print(f"[{i:02d}] EPR {row['epr_bytes']:3d} B | PSNR {row['PSNR_dB']:.2f} dB | PSNR_ROI {row['PSNR_ROI_dB']} | "
          f"SSIM {row['SSIM']:.6f} | SNR {row['SNR_dB']:.2f} dB | H(c) {row['entropy_cipher']:.4f} | NPCR {npcr_d:.2f}% | BER {row['BER']:.0f} | {v['status']}")

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "results.csv", index=False)
df.to_excel(OUT_DIR / "results.xlsx", index=False)
print("\nSemua payload lolos:", bool(df["hash_match"].all() and df["epr_match"].all() and df["signature_ok"].all()))

## Tabel hasil

In [ ]:
cols = ["payload", "epr_bytes", "embedded_bits", "bpp_image", "changed_px", "changed_px_ROI", "MSE", "PSNR_dB",
        "PSNR_ROI_dB", "SSIM", "SNR_dB", "BER", "NC", "hash_match", "signature_ok", "epr_match"]
df[cols].round(6)

In [ ]:
df[["payload", "entropy_plain", "entropy_cipher", "corrH_plain", "corrH_cipher", "corrV_cipher", "corrD_cipher",
    "NPCR_plain_vs_cipher_%", "NPCR_diff_1px_%", "UACI_diff_1px_%"]].round(6)

In [ ]:
df[["payload", "epr_bytes"] + [f"t_{k}_ms" for k in STAGES_P + STAGES_V] + ["t_sender_ms", "t_receiver_ms", "peak_mem_KB"]].round(3)

## Grafik per payload

In [ ]:
x = df["payload"]
fig, ax = plt.subplots(1, 4, figsize=(19, 4))
for a, col, title in zip(ax, ["PSNR_dB", "SSIM", "MSE", "SNR_dB"], ["PSNR seluruh citra (dB)", "SSIM", "MSE", "SNR (dB)"]):
    a.plot(x, df[col], "o-"); a.set_title(title); a.set_xlabel("Payload ke-"); a.set_xticks(x); a.grid(alpha=.3)
fig.suptitle("Kualitas citra ber-watermark (PSNR area ROI = tak hingga untuk semua payload)")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_kualitas.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x, df["epr_bytes"]); ax.set_xlabel("Payload ke-"); ax.set_ylabel("Ukuran EPR (byte)"); ax.set_xticks(x)
ax2 = ax.twinx(); ax2.plot(x, df["roni_used_%"], "o-", color="tab:red"); ax2.set_ylabel("RONI terpakai (%)")
ax.set_title("Ukuran payload & pemakaian kapasitas RONI"); fig.tight_layout()
fig.savefig(OUT_DIR / "fig_payload_capacity.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(x, df["entropy_cipher"], "o-", label="cipher"); ax[0].plot(x, df["entropy_plain"], "s--", label="watermarked")
ax[0].axhline(8, color="gray", ls=":"); ax[0].set_title("Entropi (ideal 8)"); ax[0].legend()
ax[1].plot(x, df["NPCR_diff_1px_%"], "o-", label="NPCR"); ax[1].plot(x, df["UACI_diff_1px_%"], "s-", label="UACI")
ax[1].axhline(99.6094, color="gray", ls=":"); ax[1].axhline(33.4635, color="gray", ls=":")
ax[1].set_title("NPCR & UACI diferensial 1 piksel (%)"); ax[1].legend()
for a in ax: a.set_xlabel("Payload ke-"); a.set_xticks(x); a.grid(alpha=.3)
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_enkripsi.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(9, 4)); bottom = np.zeros(len(df))
for k in STAGES_P + STAGES_V:
    ax.bar(x, df[f"t_{k}_ms"], bottom=bottom, label=k); bottom += df[f"t_{k}_ms"].values
ax.set_xlabel("Payload ke-"); ax.set_ylabel("ms"); ax.set_xticks(x); ax.set_title("Waktu proses per tahap")
ax.legend(fontsize=7, ncol=2); fig.tight_layout(); fig.savefig(OUT_DIR / "fig_waktu.png", dpi=150); plt.show()

## Visualisasi citra (payload ke-10)

In [ ]:
p10, v10 = results[10]["p"], results[10]["v"]
wm10 = p10["watermarked"]
diff = np.abs(cover.astype(np.int16) - wm10.astype(np.int16)) * 255
fig, ax = plt.subplots(1, 5, figsize=(25, 4.5))
panels = [(cover, "Cover"), (overlay_mask(cover, MASK), "RONI (biru)"), (diff, "Selisih x255 (hanya di RONI)"),
          (p10["encrypted"], "Terenkripsi AES-SIV"), (v10["image"], "Hasil dekripsi")]
for a, (im, t) in zip(ax, panels):
    a.imshow(im, cmap="gray", vmin=0, vmax=255); a.set_title(t); a.axis("off")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_citra.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, im, t in zip(ax, [cover, wm10, p10["encrypted"]], ["Cover", "Watermarked", "Encrypted"]):
    a.hist(im.ravel(), bins=256, range=(0, 256), color="k"); a.set_title(f"Histogram {t}")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_histogram.png", dpi=150); plt.show()

rng = np.random.default_rng(0)
ri = rng.integers(0, cover.shape[0], 3000); ci = rng.integers(0, cover.shape[1] - 1, 3000)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
for a, im, t in [(ax[0], wm10, "Watermarked"), (ax[1], p10["encrypted"], "Encrypted")]:
    a.scatter(im[ri, ci], im[ri, ci + 1], s=2, alpha=.5)
    a.set_title(f"Korelasi horizontal - {t}"); a.set_xlabel("p(x,y)"); a.set_ylabel("p(x,y+1)")
fig.tight_layout(); fig.savefig(OUT_DIR / "fig_korelasi.png", dpi=150); plt.show()
print("EPR hasil ekstraksi payload 10:\n", v10["epr"])

## Nilai tambah — Security
Kunci salah, ciphertext diubah, kunci dokter palsu, dan perbandingan mode AES (uji diferensial dengan key & IV sama).

In [ ]:
p10 = results[10]["p"]
tests = []
r_ = receive(p10["encrypted"], p10["meta"], KeyBundle.generate(), KEYS.public_key)
tests.append({"uji": "Dekripsi dengan kunci salah", "hasil": r_["status"], "keterangan": r_["message"]})
bad = p10["encrypted"].copy(); bad[0, 0] ^= 1
r_ = receive(bad, p10["meta"], KEYS, KEYS.public_key)
tests.append({"uji": "1 bit ciphertext diubah saat transmisi", "hasil": r_["status"], "keterangan": r_["message"]})
r_ = verify_image(p10["watermarked"], KEYS, KeyBundle.generate().public_key)
tests.append({"uji": "Verifikasi dengan kunci publik dokter lain", "hasil": r_["status"], "keterangan": r_["message"]})
fake = KeyBundle(KEYS.k_img, KEYS.k_epr, KEYS.k_wm, KeyBundle.generate().signing_key)
forged = protect(cover, payloads[-1].replace("Taharo Hafwenia", "Dokter Palsu"), fake)
r_ = verify_image(forged["watermarked"], KEYS, KEYS.public_key)
tests.append({"uji": "Pemalsu menandatangani ulang dengan kunci sendiri", "hasil": r_["status"], "keterangan": r_["message"]})
df_sec = pd.DataFrame(tests); df_sec.to_csv(OUT_DIR / "security_tests.csv", index=False)
display(df_sec)

cmp_rows = []
for m in ["siv", "ctr", "cbc"]:
    n_, u_ = differential_test(p10["watermarked"], KEYS.k_img, m)
    cmp_rows.append({"mode": f"AES-256-{m.upper()}", "NPCR_%": n_, "UACI_%": u_,
                     "integritas_bawaan": m == "siv"})
df_modes = pd.DataFrame(cmp_rows); df_modes.to_csv(OUT_DIR / "compare_modes.csv", index=False)
df_modes.round(4)

## Nilai tambah — Simulasi serangan & lokalisasi manipulasi
Serangan lokal dilakukan di area paru (ROI). TPR/FPR dihitung per blok 32×32 terhadap ground truth piksel yang benar-benar berubah.

In [ ]:
att_rows, att_imgs = [], {}
for k, desc in ATTACKS.items():
    a = attack(p10["watermarked"], k)
    vv = verify_image(a, KEYS, KEYS.public_key)
    gt = block_ground_truth(p10["watermarked"], a)
    row = {"serangan": desc, "piksel_berubah": int(np.sum(a != p10["watermarked"])),
           "status": vv["status"], "watermark_terbaca": vv["watermark_found"],
           "tanda_tangan_valid": vv["signature_ok"], "keterangan": vv["message"]}
    if vv["tamper_map"] is not None:
        row.update(localization_scores(vv["tamper_map"], gt))
    att_rows.append(row); att_imgs[k] = (a, vv)
df_att = pd.DataFrame(att_rows); df_att.to_csv(OUT_DIR / "attacks.csv", index=False)
display(df_att)

local = [k for k in ATTACKS if att_imgs[k][1]["tamper_map"] is not None]
fig, ax = plt.subplots(1, len(local), figsize=(5 * len(local), 4))
for a, k in zip(np.atleast_1d(ax), local):
    img_a, vv = att_imgs[k]
    a.imshow(tamper_overlay(img_a, vv["tamper_map"])); a.set_title(ATTACKS[k], fontsize=9); a.axis("off")
fig.suptitle("Peta manipulasi (merah = blok terdeteksi diubah)"); fig.tight_layout()
fig.savefig(OUT_DIR / "fig_serangan.png", dpi=150); plt.show()

## Nilai tambah — Capacity
EPR acak dari kecil sampai kapasitas maksimum RONI.

In [ ]:
MAX_EPR = max_epr_bytes(cover)
cap_rows = []
for frac in [0.1, 0.25, 0.5, 0.75, 1.0]:
    n = int(MAX_EPR * frac)
    txt = "X" * n
    p = protect(cover, txt, KEYS)
    v = verify_image(p["watermarked"], KEYS, KEYS.public_key)
    cap_rows.append({"epr_bytes": n, "roni_used_%": 100 * p["bits"].size / CAP_BITS,
                     "PSNR_dB": psnr(cover, p["watermarked"]), "SSIM": ssim(cover, p["watermarked"]), "SNR_dB": snr(cover, p["watermarked"]),
                     "PSNR_ROI_dB": psnr_region(cover, p["watermarked"], ROI), "status": v["status"]})
try:
    protect(cover, "X" * (MAX_EPR + 1), KEYS); over = "muat (tidak terduga)"
except WatermarkError as e:
    over = f"ditolak: {e}"
df_cap = pd.DataFrame(cap_rows); df_cap.to_csv(OUT_DIR / "capacity.csv", index=False)
print("EPR melebihi kapasitas ->", over)
df_cap.round(4)

## Simpan & download semua output (zip)

In [ ]:
shutil.make_archive("hasil_medsecure", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("hasil_medsecure.zip")
except ImportError:
    print("Tersimpan: hasil_medsecure.zip")